In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS real_estate.gold;

In [0]:
#loading cleaned data
df = spark.table('real_estate.silver.washington_ultimate')
display(df.limit(10))

###Creating Dimention Tables 

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

dim_location = df \
    .select('zip') \
    .distinct() \
    .withColumn('location_id', monotonically_increasing_id())

display(dim_location)

In [0]:
df.printSchema()

In [0]:
dim_property = df.select(
    'property_id',
    'type',
    'year_built',
    'sqft'
)

display(dim_property.limit(10))

In [0]:
dim_features = df.select(
    'property_id',
    'stories',
    'beds',
    'baths',
    'baths_full',
    'baths_full_calc',
    'garage'
)

display(dim_features.limit(10))

In [0]:
dim_discription = df.select('property_id', 'sanitized_text')

display(dim_discription.limit(10))

###Fact Table

In [0]:
fact_real_estate = df.join(dim_location, on='zip', how='left') \
    .select(
        'property_id',
        'location_id',
        'listprice',
        'lastsoldprice',
        'list_to_sold_ratio',
        'price_per_sqft'
    )

display(fact_real_estate.limit(10))

#####Loading data to tables

In [0]:
dim_location.write.format('delta').mode('overwrite').saveAsTable('real_estate.gold.dim_location')
dim_property.write.format('delta').mode('overwrite').saveAsTable('real_estate.gold.dim_property')
dim_features.write.format('delta').mode('overwrite').saveAsTable('real_estate.gold.dim_features')
dim_discription.write.format('delta').mode('overwrite').saveAsTable('real_estate.gold.dim_discription')
fact_real_estate.write.format('delta').mode('overwrite').saveAsTable('real_estate.gold.fact_real_estate')

In [0]:
display(spark.sql('select * from real_estate.gold.dim_location limit 10'))
display(spark.sql('select * from real_estate.gold.dim_features limit 10'))
display(spark.sql('select * from real_estate.gold.dim_property limit 10'))
display(spark.sql('select * from real_estate.gold.dim_discription limit 10'))
display(spark.sql('select * from real_estate.gold.fact_real_estate limit 10'))